# DermaScan AI v3 -- Login + Dashboard + AI Bot + 3D design, all in Colab

No dataset, no training, no upload needed to get this running -- it works immediately with realistic mock predictions.
Everything runs from this ONE notebook. Just run every cell top to bottom, in order.

**Pages included:** Home (3D hero animation), Login, Signup, Dashboard, Diagnose (upload+predict), About, Contact, AI Bot (chat assistant).


## Step 1 — Install libraries

In [1]:
pip install -q flask flask-cors pyngrok opencv-python-headless scikit-learn seaborn matplotlib


Note: you may need to restart the kernel to use updated packages.


## Step 2 — Create the project folders

In [2]:
import os
for p in ['dermascan/frontend/css', 'dermascan/frontend/js', 'dermascan/backend']:
    os.makedirs(p, exist_ok=True)
print("Created project folders: dermascan/frontend/css, dermascan/frontend/js, dermascan/backend")


Created project folders: dermascan/frontend/css, dermascan/frontend/js, dermascan/backend


## Step 3 — Write the frontend files
Style + 6 JS files + 8 HTML pages. No visible output is normal -- it just means the file got saved.

In [3]:
%%writefile dermascan/frontend/css/style.css
/* ===========================================================
   DermaScan AI v2 — Design tokens
   =========================================================== */
:root {
  --bg: #0a0f1c;
  --bg-elevated: #101827;
  --surface: #131d30;
  --surface-hover: #182339;
  --glass: rgba(19, 29, 48, 0.6);
  --border: #223151;
  --border-soft: rgba(255,255,255,0.06);
  --text: #eaeef6;
  --text-muted: #8d9bb5;
  --teal: #4fe8c9;
  --teal-dim: rgba(79, 232, 201, 0.14);
  --amber: #ffb970;
  --amber-dim: rgba(255, 185, 112, 0.14);
  --violet: #a78bfa;
  --violet-dim: rgba(167, 139, 250, 0.14);
  --danger: #fb7185;
  --danger-dim: rgba(251, 113, 133, 0.12);
  --font-display: 'Space Grotesk', sans-serif;
  --font-body: 'Inter', sans-serif;
  --font-mono: 'JetBrains Mono', monospace;
  --radius-sm: 10px;
  --radius: 16px;
  --radius-lg: 22px;
  --container: 1160px;
  --shadow-lg: 0 30px 80px -30px rgba(0,0,0,0.55);
}

@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after { animation-duration: 0.01ms !important; animation-iteration-count: 1 !important; transition-duration: 0.01ms !important; }
}

* { box-sizing: border-box; margin: 0; padding: 0; }
html { scroll-behavior: smooth; }
body {
  background: var(--bg);
  background-image:
    radial-gradient(circle at 12% -5%, rgba(79, 232, 201, 0.08), transparent 42%),
    radial-gradient(circle at 90% 10%, rgba(255, 185, 112, 0.06), transparent 38%),
    radial-gradient(circle at 50% 100%, rgba(167, 139, 250, 0.05), transparent 45%);
  color: var(--text);
  font-family: var(--font-body);
  line-height: 1.6;
  -webkit-font-smoothing: antialiased;
  min-height: 100vh;
}
a { color: inherit; text-decoration: none; }
img { max-width: 100%; display: block; }
ul { list-style: none; }
.container { max-width: var(--container); margin: 0 auto; padding: 0 28px; }

/* ---------- Navigation (public/marketing pages) ---------- */
.navbar { position: sticky; top: 0; z-index: 100; backdrop-filter: blur(16px); background: rgba(10, 15, 28, 0.7); border-bottom: 1px solid var(--border-soft); }
.nav-inner { display: flex; align-items: center; justify-content: space-between; padding: 18px 28px; max-width: var(--container); margin: 0 auto; }
.brand { display: flex; align-items: center; gap: 10px; font-family: var(--font-display); font-weight: 600; font-size: 1.15rem; }
.brand-mark { width: 30px; height: 30px; border-radius: 9px; background: linear-gradient(135deg, var(--teal), var(--amber)); display: grid; place-items: center; color: #06261f; font-weight: 800; font-size: 0.85rem; }
.nav-links { display: flex; align-items: center; gap: 30px; }
.nav-links a { font-size: 0.92rem; color: var(--text-muted); transition: color 0.2s; position: relative; }
.nav-links a:hover, .nav-links a.active { color: var(--text); }
.nav-links a.active::after { content: ''; position: absolute; left: 0; right: 0; bottom: -20px; height: 2px; background: var(--teal); }
.nav-cta { padding: 9px 18px; border-radius: 999px; background: var(--teal); color: #06261f !important; font-weight: 600; font-size: 0.88rem; }

/* ---------- Buttons ---------- */
.btn { display: inline-flex; align-items: center; justify-content: center; gap: 8px; padding: 13px 26px; border-radius: 999px; font-weight: 600; font-size: 0.94rem; font-family: var(--font-body); border: 1px solid transparent; cursor: pointer; transition: transform 0.15s ease, box-shadow 0.15s ease, background 0.15s; }
.btn:active { transform: scale(0.97); }
.btn-primary { background: var(--teal); color: #06261f; }
.btn-primary:hover { box-shadow: 0 0 0 6px var(--teal-dim); }
.btn-ghost { background: transparent; border-color: var(--border); color: var(--text); }
.btn-ghost:hover { border-color: var(--teal); color: var(--teal); }
.btn-block { width: 100%; }
.btn:disabled { opacity: 0.5; cursor: not-allowed; }

/* ---------- Hero (landing) ---------- */
.hero { padding: 92px 0 60px; display: grid; grid-template-columns: 1.1fr 0.9fr; gap: 56px; align-items: center; }
.eyebrow { display: inline-flex; align-items: center; gap: 8px; font-family: var(--font-mono); font-size: 0.76rem; letter-spacing: 0.08em; text-transform: uppercase; color: var(--teal); background: var(--teal-dim); border: 1px solid rgba(79,232,201,0.3); padding: 6px 14px; border-radius: 999px; margin-bottom: 22px; }
.eyebrow .dot { width: 6px; height: 6px; border-radius: 50%; background: var(--teal); animation: pulse-dot 1.8s ease-in-out infinite; }
@keyframes pulse-dot { 0%,100% { opacity: 1; } 50% { opacity: 0.3; } }
h1.headline { font-family: var(--font-display); font-size: clamp(2.4rem, 4.5vw, 3.7rem); line-height: 1.07; font-weight: 600; letter-spacing: -0.01em; }
h1.headline .accent { color: var(--teal); }
.sub { margin-top: 22px; font-size: 1.08rem; color: var(--text-muted); max-width: 46ch; }
.hero-actions { display: flex; gap: 14px; margin-top: 34px; flex-wrap: wrap; }
.hero-stats { display: flex; gap: 36px; margin-top: 48px; }
.stat-num { font-family: var(--font-mono); font-size: 1.6rem; color: var(--teal); font-weight: 600; }
.stat-label { font-size: 0.8rem; color: var(--text-muted); margin-top: 4px; }

/* ---------- Scan visual (signature element, reused on hero + auth pages) ---------- */
.scan-frame { position: relative; border-radius: var(--radius-lg); border: 1px solid var(--border); background: var(--surface); aspect-ratio: 1 / 1.05; overflow: hidden; box-shadow: var(--shadow-lg); }
.scan-frame .skin-swatch { position: absolute; inset: 0; background: radial-gradient(circle at 30% 30%, #7a4a3a 0%, transparent 45%), radial-gradient(circle at 65% 55%, #8a5a42 0%, transparent 50%), linear-gradient(160deg, #b98564, #8c5f43 60%, #6e4933); filter: saturate(0.9); }
.scan-frame .lesion { position: absolute; left: 44%; top: 40%; width: 70px; height: 56px; border-radius: 50% 45% 55% 48%; background: radial-gradient(circle at 40% 40%, #3a2318, #1c1109 70%); box-shadow: 0 0 0 3px rgba(79,232,201,0.35); }
.scan-grid { position: absolute; inset: 0; background-image: linear-gradient(rgba(79,232,201,0.08) 1px, transparent 1px), linear-gradient(90deg, rgba(79,232,201,0.08) 1px, transparent 1px); background-size: 24px 24px; }
.scan-line { position: absolute; left: 0; right: 0; height: 2px; background: linear-gradient(90deg, transparent, var(--teal), transparent); box-shadow: 0 0 16px 2px var(--teal); animation: sweep 3.2s ease-in-out infinite; }
@keyframes sweep { 0% { top: 4%; } 50% { top: 92%; } 100% { top: 4%; } }
.scan-readout { position: absolute; left: 16px; right: 16px; bottom: 16px; background: rgba(10,15,28,0.85); border: 1px solid var(--border); border-radius: 10px; padding: 12px 14px; font-family: var(--font-mono); font-size: 0.74rem; }
.scan-readout .row { display: flex; justify-content: space-between; margin-top: 4px; color: var(--text-muted); }
.scan-readout .row b { color: var(--amber); font-weight: 600; }
.corner { position: absolute; width: 18px; height: 18px; border-color: var(--teal); }
.corner.tl { top: 12px; left: 12px; border-top: 2px solid; border-left: 2px solid; }
.corner.tr { top: 12px; right: 12px; border-top: 2px solid; border-right: 2px solid; }
.corner.bl { bottom: 12px; left: 12px; border-bottom: 2px solid; border-left: 2px solid; }
.corner.br { bottom: 12px; right: 12px; border-bottom: 2px solid; border-right: 2px solid; }

/* ---------- Sections (landing/about) ---------- */
.section { padding: 80px 0; }
.section-head { max-width: 620px; margin-bottom: 48px; }
.section-tag { font-family: var(--font-mono); font-size: 0.76rem; text-transform: uppercase; letter-spacing: 0.08em; color: var(--amber); }
.section-title { font-family: var(--font-display); font-size: clamp(1.6rem, 3vw, 2.2rem); margin-top: 10px; font-weight: 600; }
.section-desc { color: var(--text-muted); margin-top: 12px; }

.steps { display: grid; grid-template-columns: repeat(3, 1fr); gap: 24px; }
.step-card { background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 28px 24px; }
.step-num { font-family: var(--font-mono); color: var(--teal); font-size: 0.8rem; border: 1px solid var(--border); width: 30px; height: 30px; border-radius: 50%; display: grid; place-items: center; margin-bottom: 18px; }
.step-card h3 { font-family: var(--font-display); font-size: 1.05rem; margin-bottom: 8px; }
.step-card p { color: var(--text-muted); font-size: 0.92rem; }

.grid-3 { display: grid; grid-template-columns: repeat(3, 1fr); gap: 20px; }
.feature-card { background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 26px; transition: border-color 0.2s, transform 0.2s; }
.feature-card:hover { border-color: rgba(79,232,201,0.35); transform: translateY(-3px); }
.feature-icon { width: 40px; height: 40px; border-radius: 10px; background: var(--teal-dim); display: grid; place-items: center; margin-bottom: 16px; color: var(--teal); }
.feature-card h3 { font-family: var(--font-display); font-size: 1rem; margin-bottom: 8px; }
.feature-card p { color: var(--text-muted); font-size: 0.9rem; }

.stack-strip { display: flex; flex-wrap: wrap; gap: 12px; }
.stack-pill { font-family: var(--font-mono); font-size: 0.82rem; padding: 9px 16px; border: 1px solid var(--border); border-radius: 999px; color: var(--text-muted); background: var(--bg-elevated); }

.arch-diagram { display: flex; align-items: center; gap: 10px; flex-wrap: wrap; padding: 28px; background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); }
.arch-block { font-family: var(--font-mono); font-size: 0.78rem; padding: 12px 14px; border-radius: 8px; background: var(--bg-elevated); border: 1px solid var(--border); text-align: center; color: var(--text-muted); min-width: 90px; }
.arch-block b { display: block; color: var(--text); font-family: var(--font-body); font-size: 0.82rem; margin-bottom: 2px; }
.arch-arrow { color: var(--teal); font-size: 1rem; }

.metric-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; }
.metric-card { background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 22px; text-align: center; }
.metric-card .num { font-family: var(--font-mono); font-size: 1.7rem; color: var(--teal); font-weight: 600; }
.metric-card .lbl { font-size: 0.82rem; color: var(--text-muted); margin-top: 6px; }

.timeline { border-left: 2px solid var(--border); padding-left: 24px; display: flex; flex-direction: column; gap: 28px; }
.timeline-item { position: relative; }
.timeline-item::before { content: ''; position: absolute; left: -30px; top: 4px; width: 10px; height: 10px; border-radius: 50%; background: var(--teal); box-shadow: 0 0 0 4px var(--teal-dim); }
.timeline-item h4 { font-family: var(--font-display); font-size: 1rem; margin-bottom: 4px; }
.timeline-item p { color: var(--text-muted); font-size: 0.9rem; }

.team-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 18px; }
.team-card { background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 22px; text-align: center; }
.team-avatar { width: 56px; height: 56px; border-radius: 50%; margin: 0 auto 14px; background: linear-gradient(135deg, var(--teal), var(--amber)); display: grid; place-items: center; font-family: var(--font-display); font-weight: 700; color: #06261f; }
.team-card h4 { font-size: 0.94rem; }
.team-card span { font-size: 0.78rem; color: var(--text-muted); }

footer { border-top: 1px solid var(--border-soft); padding: 40px 0; margin-top: 40px; }
.footer-inner { display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 16px; }
.footer-inner p { color: var(--text-muted); font-size: 0.85rem; }
.footer-links { display: flex; gap: 22px; }
.footer-links a { color: var(--text-muted); font-size: 0.85rem; }
.footer-links a:hover { color: var(--teal); }

/* ===========================================================
   AUTH PAGES (login / signup) — split screen
   =========================================================== */
.auth-shell { min-height: 100vh; display: grid; grid-template-columns: 1fr 1fr; }
.auth-visual { position: relative; padding: 48px; display: flex; flex-direction: column; justify-content: space-between; background: var(--bg-elevated); border-right: 1px solid var(--border-soft); }
.auth-visual .brand { font-size: 1.2rem; }
.auth-visual .pitch { max-width: 380px; }
.auth-visual .pitch h2 { font-family: var(--font-display); font-size: 1.7rem; margin-bottom: 12px; line-height: 1.2; }
.auth-visual .pitch p { color: var(--text-muted); font-size: 0.95rem; }
.auth-form-side { display: flex; align-items: center; justify-content: center; padding: 40px; }
.auth-card { width: 100%; max-width: 400px; }
.auth-card .kicker { font-family: var(--font-mono); font-size: 0.76rem; text-transform: uppercase; letter-spacing: 0.08em; color: var(--teal); margin-bottom: 10px; }
.auth-card h1 { font-family: var(--font-display); font-size: 1.9rem; margin-bottom: 8px; }
.auth-card .switcher { color: var(--text-muted); font-size: 0.92rem; margin-bottom: 32px; }
.auth-card .switcher a { color: var(--teal); font-weight: 600; }
.auth-error { display: none; background: var(--danger-dim); border: 1px solid rgba(251,113,133,0.35); color: var(--danger); font-size: 0.85rem; padding: 10px 14px; border-radius: 8px; margin-bottom: 18px; }

@media (max-width: 900px) {
  .auth-shell { grid-template-columns: 1fr; }
  .auth-visual { display: none; }
}

/* ---------- Form fields (shared: auth + contact) ---------- */
.field { margin-bottom: 18px; }
.field label { display: block; font-size: 0.85rem; margin-bottom: 8px; color: var(--text-muted); }
.field input, .field textarea { width: 100%; padding: 12px 14px; border-radius: var(--radius-sm); background: var(--bg-elevated); border: 1px solid var(--border); color: var(--text); font-family: var(--font-body); font-size: 0.92rem; resize: vertical; transition: border-color 0.15s; }
.field input:focus, .field textarea:focus { outline: none; border-color: var(--teal); }
.form-status { margin-top: 14px; font-size: 0.86rem; color: var(--teal); display: none; }

/* ===========================================================
   DASHBOARD SHELL (sidebar + topbar) — protected pages
   =========================================================== */
.app-shell { display: grid; grid-template-columns: 250px 1fr; min-height: 100vh; }
.sidebar { background: var(--bg-elevated); border-right: 1px solid var(--border-soft); padding: 26px 20px; display: flex; flex-direction: column; }
.sidebar .brand { margin-bottom: 40px; padding-left: 6px; }
.side-nav { display: flex; flex-direction: column; gap: 4px; flex: 1; }
.side-nav a { display: flex; align-items: center; gap: 12px; padding: 11px 14px; border-radius: var(--radius-sm); color: var(--text-muted); font-size: 0.92rem; font-weight: 500; transition: background 0.15s, color 0.15s; }
.side-nav a .ic { width: 18px; text-align: center; }
.side-nav a:hover { background: var(--surface); color: var(--text); }
.side-nav a.active { background: var(--teal-dim); color: var(--teal); }
.sidebar-footer { border-top: 1px solid var(--border-soft); padding-top: 16px; }
.sidebar-user { display: flex; align-items: center; gap: 10px; margin-bottom: 12px; }
.sidebar-user .avatar { width: 34px; height: 34px; border-radius: 50%; background: linear-gradient(135deg, var(--teal), var(--violet)); display: grid; place-items: center; font-family: var(--font-display); font-weight: 700; font-size: 0.85rem; color: #06261f; }
.sidebar-user .name { font-size: 0.88rem; font-weight: 600; }
.sidebar-user .email { font-size: 0.75rem; color: var(--text-muted); }
.logout-link { display: flex; align-items: center; gap: 10px; padding: 10px 14px; border-radius: var(--radius-sm); color: var(--text-muted); font-size: 0.88rem; cursor: pointer; transition: background 0.15s, color 0.15s; }
.logout-link:hover { background: var(--danger-dim); color: var(--danger); }

.app-main { padding: 36px 40px; overflow-x: hidden; }
.app-topbar { display: flex; justify-content: space-between; align-items: flex-start; margin-bottom: 32px; flex-wrap: wrap; gap: 16px; }
.app-topbar h1 { font-family: var(--font-display); font-size: 1.6rem; }
.app-topbar p { color: var(--text-muted); font-size: 0.92rem; margin-top: 4px; }

.stat-cards { display: grid; grid-template-columns: repeat(4, 1fr); gap: 18px; margin-bottom: 28px; }
.stat-card { background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 22px; }
.stat-card .top { display: flex; justify-content: space-between; align-items: center; margin-bottom: 14px; }
.stat-card .ic { width: 34px; height: 34px; border-radius: 9px; display: grid; place-items: center; font-size: 0.95rem; }
.stat-card .ic.teal { background: var(--teal-dim); color: var(--teal); }
.stat-card .ic.amber { background: var(--amber-dim); color: var(--amber); }
.stat-card .ic.violet { background: var(--violet-dim); color: var(--violet); }
.stat-card .ic.danger { background: var(--danger-dim); color: var(--danger); }
.stat-card .value { font-family: var(--font-mono); font-size: 1.7rem; font-weight: 600; }
.stat-card .label { font-size: 0.82rem; color: var(--text-muted); margin-top: 4px; }

.dash-grid { display: grid; grid-template-columns: 1.1fr 1.4fr; gap: 20px; }
.panel { background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 26px; }
.panel h3 { font-family: var(--font-display); font-size: 1.05rem; margin-bottom: 20px; }

/* Donut chart (pure SVG, no chart library) */
.donut-wrap { display: flex; align-items: center; gap: 24px; }
.donut-legend { display: flex; flex-direction: column; gap: 10px; flex: 1; }
.legend-row { display: flex; align-items: center; gap: 10px; font-size: 0.85rem; }
.legend-swatch { width: 10px; height: 10px; border-radius: 3px; flex-shrink: 0; }
.legend-row .pct { margin-left: auto; font-family: var(--font-mono); color: var(--text-muted); }

.scan-list { display: flex; flex-direction: column; gap: 4px; }
.scan-row { display: flex; align-items: center; gap: 14px; padding: 12px 8px; border-radius: var(--radius-sm); transition: background 0.15s; }
.scan-row:hover { background: var(--bg-elevated); }
.scan-row .dot-ic { width: 36px; height: 36px; border-radius: 10px; display: grid; place-items: center; font-family: var(--font-mono); font-size: 0.7rem; font-weight: 700; flex-shrink: 0; }
.scan-row .info { flex: 1; min-width: 0; }
.scan-row .info .cond { font-size: 0.9rem; font-weight: 600; }
.scan-row .info .time { font-size: 0.78rem; color: var(--text-muted); }
.scan-row .conf { font-family: var(--font-mono); font-size: 0.85rem; color: var(--text-muted); }
.empty-note { color: var(--text-muted); font-size: 0.88rem; padding: 20px 0; text-align: center; }

/* ---------- Predict page (inside dashboard shell) ---------- */
.predict-layout { display: grid; grid-template-columns: 1fr 1fr; gap: 24px; align-items: start; }
.upload-card { background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 30px; }
.dropzone { border: 2px dashed var(--border); border-radius: var(--radius-sm); padding: 46px 20px; text-align: center; cursor: pointer; transition: border-color 0.2s, background 0.2s; position: relative; }
.dropzone.drag-over { border-color: var(--teal); background: var(--teal-dim); }
.dropzone svg { color: var(--teal); margin-bottom: 14px; }
.dropzone .hint { color: var(--text-muted); font-size: 0.85rem; margin-top: 6px; }
.dropzone input[type="file"] { position: absolute; inset: 0; opacity: 0; cursor: pointer; }
.preview-wrap { position: relative; border-radius: var(--radius-sm); overflow: hidden; margin-top: 20px; display: none; }
.preview-wrap img { width: 100%; max-height: 300px; object-fit: cover; }
.preview-wrap .scan-line { display: none; }
.preview-wrap.scanning .scan-line { display: block; animation: sweep 1.4s linear infinite; }

.result-card { background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 30px; min-height: 320px; }
.result-empty { color: var(--text-muted); font-size: 0.92rem; display: flex; flex-direction: column; align-items: center; justify-content: center; height: 100%; min-height: 260px; text-align: center; gap: 10px; }
.result-empty svg { color: var(--border); }
.top-result { display: flex; align-items: center; justify-content: space-between; padding-bottom: 20px; margin-bottom: 20px; border-bottom: 1px solid var(--border-soft); }
.top-result .label { font-family: var(--font-display); font-size: 1.4rem; }
.top-result .confidence { font-family: var(--font-mono); color: var(--teal); font-size: 1.3rem; font-weight: 600; }
.flag-note { background: var(--danger-dim); border: 1px solid rgba(251,113,133,0.35); color: var(--danger); font-size: 0.82rem; padding: 10px 14px; border-radius: 8px; margin-bottom: 18px; }
.bar-row { margin-bottom: 14px; }
.bar-row-head { display: flex; justify-content: space-between; font-size: 0.86rem; margin-bottom: 6px; }
.bar-row-head span:last-child { font-family: var(--font-mono); color: var(--text-muted); }
.bar-track { height: 8px; background: var(--bg-elevated); border-radius: 999px; overflow: hidden; }
.bar-fill { height: 100%; border-radius: 999px; background: linear-gradient(90deg, var(--teal), var(--amber)); width: 0%; transition: width 1s cubic-bezier(0.22, 1, 0.36, 1); }
.disclaimer { margin-top: 24px; font-size: 0.8rem; color: var(--text-muted); border-top: 1px solid var(--border-soft); padding-top: 16px; }
.saved-note { margin-top: 16px; font-size: 0.82rem; color: var(--teal); }

/* ---------- Contact layout ---------- */
.contact-layout { display: grid; grid-template-columns: 0.9fr 1.1fr; gap: 40px; }
.info-list { display: flex; flex-direction: column; gap: 20px; margin-top: 28px; }
.info-item { display: flex; gap: 14px; align-items: flex-start; }
.info-item .ic { width: 38px; height: 38px; border-radius: 10px; background: var(--amber-dim); color: var(--amber); display: grid; place-items: center; flex-shrink: 0; }
.info-item h4 { font-size: 0.95rem; margin-bottom: 2px; }
.info-item p { color: var(--text-muted); font-size: 0.88rem; }
.form-card { background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 32px; }

/* ---------- Focus + responsive ---------- */
a:focus-visible, button:focus-visible, input:focus-visible, textarea:focus-visible { outline: 2px solid var(--teal); outline-offset: 2px; }

@media (max-width: 980px) {
  .app-shell { grid-template-columns: 1fr; }
  .sidebar { display: none; }
  .dash-grid { grid-template-columns: 1fr; }
  .stat-cards { grid-template-columns: 1fr 1fr; }
}
@media (max-width: 860px) {
  .hero { grid-template-columns: 1fr; padding-top: 40px; }
  .nav-links { display: none; }
  .steps, .grid-3, .metric-grid, .team-grid { grid-template-columns: 1fr 1fr; }
  .predict-layout, .contact-layout { grid-template-columns: 1fr; }
  .hero-stats { gap: 24px; flex-wrap: wrap; }
}
@media (max-width: 520px) {
  .steps, .grid-3, .metric-grid, .team-grid, .stat-cards { grid-template-columns: 1fr; }
}

/* ===========================================================
   3D hero + tilt cards (added for v3 upgrade)
   =========================================================== */
#hero3d { width: 100%; height: 380px; border-radius: var(--radius-lg); background: radial-gradient(circle at 30% 20%, #101827, #070a12); border: 1px solid var(--border); }
.tilt-card { transform-style: preserve-3d; transition: transform 0.15s ease-out; perspective: 800px; }

/* AI Bot chat page */
.chat-shell { max-width: 720px; margin: 0 auto; display: flex; flex-direction: column; height: calc(100vh - 160px); }
.chat-window { flex: 1; overflow-y: auto; background: var(--surface); border: 1px solid var(--border-soft); border-radius: var(--radius); padding: 24px; display: flex; flex-direction: column; gap: 14px; margin-bottom: 16px; }
.chat-msg { max-width: 78%; padding: 12px 16px; border-radius: 14px; font-size: 0.92rem; line-height: 1.5; }
.chat-msg.bot { background: var(--bg-elevated); align-self: flex-start; border: 1px solid var(--border); }
.chat-msg.user { background: var(--teal); color: #06261f; align-self: flex-end; font-weight: 500; }
.chat-input-row { display: flex; gap: 12px; }
.chat-input-row input { flex: 1; padding: 14px 18px; border-radius: 999px; background: var(--surface); border: 1px solid var(--border); color: var(--text); font-size: 0.94rem; }
.chat-input-row input:focus { outline: none; border-color: var(--teal); }
.chat-suggestions { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 14px; }
.chat-chip { font-size: 0.8rem; padding: 7px 14px; border-radius: 999px; border: 1px solid var(--border); color: var(--text-muted); cursor: pointer; transition: 0.15s; }
.chat-chip:hover { border-color: var(--teal); color: var(--teal); }


Overwriting dermascan/frontend/css/style.css


In [4]:
%%writefile dermascan/frontend/js/site.js
// ===========================================================
// site.js — used on public pages (index, about, contact)
// ===========================================================
const API_BASE = ""; // same origin — Flask serves both the site and the API

// Swap "Log in" nav link for "Dashboard" if a session is already active
(async function checkAuthForNav() {
  const link = document.getElementById("authNavLink");
  if (!link) return;
  try {
    const res = await fetch(`${API_BASE}/api/me`, { credentials: "include" });
    if (res.ok) {
      link.textContent = "Dashboard";
      link.href = "dashboard.html";
    }
  } catch (e) {
    // backend not reachable yet — leave default "Log in" link
  }
})();

// Contact form (public, no auth required)
const contactForm = document.getElementById("contactForm");
if (contactForm) {
  contactForm.addEventListener("submit", async e => {
    e.preventDefault();
    const payload = {
      name: document.getElementById("name").value,
      email: document.getElementById("email").value,
      message: document.getElementById("message").value
    };
    const status = document.getElementById("formStatus");
    try {
      await fetch(`${API_BASE}/api/contact`, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(payload)
      });
    } catch (err) {
      console.warn("Contact backend not reachable.", err);
    }
    status.style.display = "block";
    contactForm.reset();
  });
}


Overwriting dermascan/frontend/js/site.js


In [5]:
%%writefile dermascan/frontend/js/auth.js
// ===========================================================
// auth.js — used on login.html and signup.html
// ===========================================================
const API_BASE = "";

function showAuthError(message) {
  const el = document.getElementById("authError");
  el.textContent = message;
  el.style.display = "block";
}

const loginForm = document.getElementById("loginForm");
if (loginForm) {
  loginForm.addEventListener("submit", async e => {
    e.preventDefault();
    const btn = document.getElementById("loginBtn");
    btn.disabled = true;
    btn.textContent = "Logging in...";

    const payload = {
      email: document.getElementById("email").value.trim(),
      password: document.getElementById("password").value
    };

    try {
      const res = await fetch(`${API_BASE}/api/login`, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        credentials: "include",
        body: JSON.stringify(payload)
      });
      const data = await res.json();
      if (!res.ok) throw new Error(data.error || "Login failed");
      window.location.href = "dashboard.html";
    } catch (err) {
      showAuthError(err.message);
      btn.disabled = false;
      btn.textContent = "Log in";
    }
  });
}

const signupForm = document.getElementById("signupForm");
if (signupForm) {
  signupForm.addEventListener("submit", async e => {
    e.preventDefault();
    const btn = document.getElementById("signupBtn");

    const password = document.getElementById("password").value;
    const confirmPassword = document.getElementById("confirmPassword").value;
    if (password !== confirmPassword) {
      showAuthError("Passwords don't match.");
      return;
    }

    btn.disabled = true;
    btn.textContent = "Creating account...";

    const payload = {
      name: document.getElementById("name").value.trim(),
      email: document.getElementById("email").value.trim(),
      password
    };

    try {
      const res = await fetch(`${API_BASE}/api/signup`, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        credentials: "include",
        body: JSON.stringify(payload)
      });
      const data = await res.json();
      if (!res.ok) throw new Error(data.error || "Sign up failed");
      window.location.href = "dashboard.html";
    } catch (err) {
      showAuthError(err.message);
      btn.disabled = false;
      btn.textContent = "Create account";
    }
  });
}


Overwriting dermascan/frontend/js/auth.js


In [6]:
%%writefile dermascan/frontend/js/guard.js
// ===========================================================
// guard.js — protects dashboard.html and predict.html
// Redirects to login.html if there's no active session, and
// fills in the sidebar user card once auth is confirmed.
// ===========================================================
const API_BASE = "";
let CURRENT_USER = null;

async function requireAuth() {
  try {
    const res = await fetch(`${API_BASE}/api/me`, { credentials: "include" });
    if (!res.ok) throw new Error("not logged in");
    const data = await res.json();
    CURRENT_USER = data.user;
    renderUserCard(data.user);
    return data.user;
  } catch (e) {
    window.location.href = "login.html";
    return null;
  }
}

function renderUserCard(user) {
  const nameEl = document.getElementById("userName");
  const emailEl = document.getElementById("userEmail");
  const avatarEl = document.getElementById("userAvatar");
  const greetingEl = document.getElementById("greeting");
  if (nameEl) nameEl.textContent = user.name;
  if (emailEl) emailEl.textContent = user.email;
  if (avatarEl) avatarEl.textContent = user.name.charAt(0).toUpperCase();
  if (greetingEl) greetingEl.textContent = `Welcome back, ${user.name.split(" ")[0]}`;
}

const logoutBtn = document.getElementById("logoutBtn");
if (logoutBtn) {
  logoutBtn.addEventListener("click", async () => {
    await fetch(`${API_BASE}/api/logout`, { method: "POST", credentials: "include" });
    window.location.href = "login.html";
  });
}

// Run immediately on every protected page that includes this script
requireAuth();


Overwriting dermascan/frontend/js/guard.js


In [7]:
%%writefile dermascan/frontend/js/dashboard.js
// ===========================================================
// dashboard.js — used on dashboard.html
// ===========================================================
const CONDITION_COLORS = {
  Eczema: "#4fe8c9",
  Melanoma: "#fb7185",
  Acne: "#ffb970",
  Psoriasis: "#a78bfa"
};

async function loadDashboard() {
  await requireAuth(); // from guard.js — redirects to login if not authed

  try {
    const res = await fetch(`${API_BASE}/api/dashboard/stats`, { credentials: "include" });
    if (!res.ok) throw new Error("Could not load dashboard stats");
    const data = await res.json();
    renderStatCards(data);
    renderDonut(data.condition_counts);
    renderRecentScans(data.recent);
  } catch (err) {
    console.error(err);
    document.getElementById("recentScans").innerHTML =
      `<p class="empty-note">Could not reach the backend. Make sure the Flask server is running.</p>`;
  }
}

function renderStatCards(data) {
  document.getElementById("statTotal").textContent = data.total_scans;
  document.getElementById("statTop").textContent = data.most_common || "—";
  document.getElementById("statAvgConf").textContent =
    data.total_scans ? `${(data.avg_confidence * 100).toFixed(0)}%` : "—";
  document.getElementById("statFlagged").textContent = data.melanoma_flags;
}

function renderDonut(counts) {
  const container = document.getElementById("donutContainer");
  const total = Object.values(counts).reduce((a, b) => a + b, 0);

  if (!total) {
    container.innerHTML = `<p class="empty-note">No scans yet — run your first analysis to see a breakdown here.</p>`;
    return;
  }

  const size = 160, stroke = 22, radius = (size - stroke) / 2, circumference = 2 * Math.PI * radius;
  let offset = 0;
  let circles = "";
  let legend = "";

  Object.entries(counts).forEach(([label, count]) => {
    if (!count) return;
    const fraction = count / total;
    const dash = fraction * circumference;
    const color = CONDITION_COLORS[label] || "#8d9bb5";
    circles += `<circle cx="${size/2}" cy="${size/2}" r="${radius}" fill="none" stroke="${color}"
      stroke-width="${stroke}" stroke-dasharray="${dash} ${circumference - dash}"
      stroke-dashoffset="${-offset}" transform="rotate(-90 ${size/2} ${size/2})" />`;
    offset += dash;
    legend += `<div class="legend-row"><span class="legend-swatch" style="background:${color}"></span>
      ${label} <span class="pct">${Math.round(fraction * 100)}%</span></div>`;
  });

  container.innerHTML = `
    <div class="donut-wrap">
      <svg width="${size}" height="${size}" viewBox="0 0 ${size} ${size}">
        <circle cx="${size/2}" cy="${size/2}" r="${radius}" fill="none" stroke="#182339" stroke-width="${stroke}" />
        ${circles}
      </svg>
      <div class="donut-legend">${legend}</div>
    </div>
  `;
}

function renderRecentScans(scans) {
  const list = document.getElementById("recentScans");
  if (!scans || !scans.length) {
    list.innerHTML = `<p class="empty-note">No scans yet — click "New scan" to analyze your first image.</p>`;
    return;
  }

  list.innerHTML = scans.map(scan => {
    const color = CONDITION_COLORS[scan.prediction] || "#8d9bb5";
    const initials = scan.prediction.substring(0, 2).toUpperCase();
    const date = new Date(scan.created_at).toLocaleString();
    return `
      <div class="scan-row">
        <div class="dot-ic" style="background:${color}22; color:${color};">${initials}</div>
        <div class="info">
          <div class="cond">${scan.prediction}</div>
          <div class="time">${date}</div>
        </div>
        <div class="conf">${(scan.confidence * 100).toFixed(1)}%</div>
      </div>
    `;
  }).join("");
}

loadDashboard();


Overwriting dermascan/frontend/js/dashboard.js


In [8]:
%%writefile dermascan/frontend/js/predict.js
// ===========================================================
// predict.js — used on predict.html (protected page)
// ===========================================================
requireAuth(); // from guard.js

const dropzone = document.getElementById("dropzone");
const fileInput = document.getElementById("fileInput");
const previewWrap = document.getElementById("previewWrap");
const previewImg = document.getElementById("previewImg");
const analyzeBtn = document.getElementById("analyzeBtn");
const resetBtn = document.getElementById("resetBtn");
const resultEmpty = document.getElementById("resultEmpty");
const resultContent = document.getElementById("resultContent");

let selectedFile = null;

["dragenter", "dragover"].forEach(evt =>
  dropzone.addEventListener(evt, e => { e.preventDefault(); dropzone.classList.add("drag-over"); })
);
["dragleave", "drop"].forEach(evt =>
  dropzone.addEventListener(evt, e => { e.preventDefault(); dropzone.classList.remove("drag-over"); })
);
dropzone.addEventListener("drop", e => {
  const file = e.dataTransfer.files[0];
  if (file) handleFile(file);
});
fileInput.addEventListener("change", e => {
  const file = e.target.files[0];
  if (file) handleFile(file);
});

function handleFile(file) {
  if (!file.type.startsWith("image/")) {
    alert("Please upload an image file (JPG or PNG).");
    return;
  }
  selectedFile = file;
  const reader = new FileReader();
  reader.onload = e => {
    previewImg.src = e.target.result;
    previewWrap.style.display = "block";
    analyzeBtn.disabled = false;
  };
  reader.readAsDataURL(file);
}

resetBtn.addEventListener("click", () => {
  selectedFile = null;
  fileInput.value = "";
  previewWrap.style.display = "none";
  previewWrap.classList.remove("scanning");
  analyzeBtn.disabled = true;
  resultEmpty.style.display = "flex";
  resultContent.style.display = "none";
  resultContent.innerHTML = "";
});

analyzeBtn.addEventListener("click", async () => {
  if (!selectedFile) return;

  analyzeBtn.disabled = true;
  analyzeBtn.textContent = "Analyzing...";
  previewWrap.classList.add("scanning");

  const formData = new FormData();
  formData.append("image", selectedFile);

  try {
    const res = await fetch(`${API_BASE}/api/predict`, {
      method: "POST",
      credentials: "include", // sends the session cookie so the backend can save this scan to your history
      body: formData
    });
    if (res.status === 401) {
      window.location.href = "login.html";
      return;
    }
    if (!res.ok) throw new Error("Prediction request failed");
    const data = await res.json();
    renderResult(data);
  } catch (err) {
    renderError(err);
  } finally {
    previewWrap.classList.remove("scanning");
    analyzeBtn.disabled = false;
    analyzeBtn.textContent = "Analyze image";
  }
});

function renderResult(data) {
  resultEmpty.style.display = "none";
  resultContent.style.display = "block";

  const sorted = Object.entries(data.probabilities).sort((a, b) => b[1] - a[1]);
  const isMelanoma = data.prediction.toLowerCase() === "melanoma";

  let html = `
    <div class="top-result">
      <div>
        <div style="color:var(--text-muted); font-size:0.82rem; margin-bottom:4px;">Predicted condition</div>
        <div class="label">${data.prediction}</div>
      </div>
      <div class="confidence">${(data.confidence * 100).toFixed(1)}%</div>
    </div>
  `;

  if (isMelanoma) {
    html += `<div class="flag-note">⚠ Melanoma indicators detected — please consult a dermatologist for a clinical diagnosis.</div>`;
  }

  sorted.forEach(([label, prob]) => {
    const pct = (prob * 100).toFixed(1);
    html += `
      <div class="bar-row">
        <div class="bar-row-head"><span>${label}</span><span>${pct}%</span></div>
        <div class="bar-track"><div class="bar-fill" data-pct="${pct}"></div></div>
      </div>
    `;
  });

  html += `<p class="disclaimer">This is an AI-generated first read, not a medical diagnosis. Always confirm with a licensed dermatologist.</p>`;
  html += `<p class="saved-note">✓ Saved to your dashboard history.</p>`;

  resultContent.innerHTML = html;

  requestAnimationFrame(() => {
    resultContent.querySelectorAll(".bar-fill").forEach(bar => {
      bar.style.width = bar.dataset.pct + "%";
    });
  });
}

function renderError(err) {
  resultEmpty.style.display = "none";
  resultContent.style.display = "block";
  resultContent.innerHTML = `
    <div class="flag-note">
      Could not reach the prediction server. Make sure the Flask backend is running, then try again.
    </div>
  `;
  console.error(err);
}


Overwriting dermascan/frontend/js/predict.js


In [9]:
%%writefile dermascan/frontend/js/chatbot.js
// ===========================================================
// chatbot.js — simple rule-based AI Bot (no API key needed)
// Matches keywords in the question to a canned, informative answer.
// ===========================================================
const KNOWLEDGE_BASE = [
  {
    keywords: ["eczema"],
    answer: "Eczema (atopic dermatitis) causes dry, itchy, inflamed patches of skin, often in skin folds like elbows and knees. It's a chronic condition often linked to allergies or a sensitive skin barrier. DermaScan can flag it from an image, but a dermatologist should confirm and guide treatment."
  },
  {
    keywords: ["melanoma", "cancer", "warning sign", "abcde"],
    answer: "Melanoma is the most serious type of skin cancer. Common warning signs follow the 'ABCDE' rule: Asymmetry, Border irregularity, Color variation, Diameter over 6mm, and Evolving size/shape. If DermaScan flags melanoma, please see a dermatologist promptly — early detection matters a lot here."
  },
  {
    keywords: ["acne"],
    answer: "Acne happens when hair follicles get clogged with oil and dead skin cells, causing pimples, blackheads or cysts — most common on the face, chest and back. It's usually manageable with topical treatments; a dermatologist can help with persistent or severe cases."
  },
  {
    keywords: ["psoriasis"],
    answer: "Psoriasis is a chronic autoimmune condition that speeds up skin cell turnover, causing thick, scaly, red or silvery patches — often on the scalp, elbows and knees. It tends to flare and settle over time and is managed rather than cured."
  },
  {
    keywords: ["how does the model work", "model work", "architecture", "cnn", "how it works"],
    answer: "DermaScan uses a Convolutional Neural Network built on MobileNetV2 (transfer learning). An uploaded image is resized to 224x224, passed through the pretrained MobileNetV2 base to extract features, then a small classification head predicts probabilities across Eczema, Melanoma, Acne and Psoriasis."
  },
  {
    keywords: ["accurate", "accuracy", "how good", "reliable"],
    answer: "Accuracy depends on the training data and how many epochs the model was trained for — check the confusion matrix and classification report generated during training for real numbers. Either way, DermaScan is meant as a first-read assistant, not a replacement for a dermatologist's diagnosis."
  },
  {
    keywords: ["dataset", "ham10000", "training data"],
    answer: "The reference dataset is HAM10000 ('Human Against Machine with 10000 training images'), a public benchmark of labeled dermoscopic images. Note it has 7 lesion-type labels rather than Eczema/Acne/Psoriasis directly, so a matching or combined dataset is used for those classes."
  },
  {
    keywords: ["dashboard", "history", "scans", "track"],
    answer: "Once you sign up and log in, every scan you run on the Diagnose page is saved automatically. Your Dashboard shows total scans, a condition breakdown, average confidence, and your most recent results."
  },
  {
    keywords: ["login", "signup", "account", "sign up"],
    answer: "You can create a free account from the Sign up page — it just needs your name, email and a password. Once logged in, your scan history is tracked on your personal Dashboard."
  },
  {
    keywords: ["team", "who made", "bug slayers", "college", "rec"],
    answer: "This project was built by team Bug Slayers, AIDS department, Rajalakshmi Engineering College, Chennai, as an internship project."
  },
  {
    keywords: ["hi", "hello", "hey"],
    answer: "Hey! Ask me anything about eczema, melanoma, acne, psoriasis, or how DermaScan AI works."
  }
];

const FALLBACK_ANSWER = "I don't have a specific answer for that yet — try asking about eczema, melanoma, acne, psoriasis, the model, the dataset, or your dashboard. For anything about your own skin, please consult a dermatologist.";

function findAnswer(question) {
  const q = question.toLowerCase();
  for (const entry of KNOWLEDGE_BASE) {
    if (entry.keywords.some(k => q.includes(k))) {
      return entry.answer;
    }
  }
  return FALLBACK_ANSWER;
}

const chatWindow = document.getElementById("chatWindow");
const chatInput = document.getElementById("chatInput");
const sendBtn = document.getElementById("sendBtn");

function addMessage(text, sender) {
  const div = document.createElement("div");
  div.className = `chat-msg ${sender}`;
  div.textContent = text;
  chatWindow.appendChild(div);
  chatWindow.scrollTop = chatWindow.scrollHeight;
}

function handleSend(question) {
  const text = (question || chatInput.value).trim();
  if (!text) return;
  addMessage(text, "user");
  chatInput.value = "";
  setTimeout(() => {
    addMessage(findAnswer(text), "bot");
  }, 400); // small delay so it feels like it's "thinking"
}

sendBtn.addEventListener("click", () => handleSend());
chatInput.addEventListener("keydown", e => {
  if (e.key === "Enter") handleSend();
});
document.querySelectorAll(".chat-chip").forEach(chip => {
  chip.addEventListener("click", () => handleSend(chip.dataset.q));
});


Overwriting dermascan/frontend/js/chatbot.js


In [10]:
%%writefile dermascan/frontend/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>DermaScan AI — Skin Disease Classification</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="css/style.css">
</head>
<body>

<nav class="navbar">
  <div class="nav-inner">
    <a href="index.html" class="brand"><span class="brand-mark">DS</span> DermaScan AI</a>
    <div class="nav-links">
      <a href="index.html" class="active">Home</a>
      <a href="about.html">About</a>
      <a href="chatbot.html">AI Bot</a>
      <a href="contact.html">Contact</a>
      <a href="login.html" class="nav-cta" id="authNavLink">Log in</a>
    </div>
  </div>
</nav>

<header class="container hero">
  <div>
    <span class="eyebrow"><span class="dot"></span> CNN-powered dermoscopic analysis</span>
    <h1 class="headline">Detect skin conditions <span class="accent">before they escalate.</span></h1>
    <p class="sub">DermaScan AI classifies eczema, melanoma, acne and psoriasis from a single dermoscopic image using a deep convolutional neural network — with a personal dashboard and an AI assistant to answer your questions.</p>
    <div class="hero-actions">
      <a href="signup.html" class="btn btn-primary">Create free account →</a>
      <a href="chatbot.html" class="btn btn-ghost">Ask the AI Bot</a>
    </div>
    <div class="hero-stats">
      <div><div class="stat-num">4</div><div class="stat-label">Conditions classified</div></div>
      <div><div class="stat-num">10,015</div><div class="stat-label">Training images</div></div>
      <div><div class="stat-num">&lt;2s</div><div class="stat-label">Prediction time</div></div>
    </div>
  </div>

  <canvas id="hero3d"></canvas>
</header>

<section class="section container">
  <div class="section-head">
    <span class="section-tag">Workflow</span>
    <h2 class="section-title">From photo to prediction in three steps</h2>
    <p class="section-desc">Sign up once — every scan after that is saved to your personal dashboard.</p>
  </div>
  <div class="steps">
    <div class="step-card"><div class="step-num">01</div><h3>Create an account</h3><p>A free account keeps a private history of every image you analyze.</p></div>
    <div class="step-card"><div class="step-num">02</div><h3>Upload & analyze</h3><p>The image is pre-processed and passed through a trained CNN.</p></div>
    <div class="step-card"><div class="step-num">03</div><h3>Track it on your dashboard</h3><p>See every past scan, condition breakdown, and confidence trend in one place.</p></div>
  </div>
</section>

<section class="section container" style="background:var(--bg-elevated); border-radius:24px;">
  <div class="section-head">
    <span class="section-tag">Capabilities</span>
    <h2 class="section-title">Built for accuracy, scale and privacy</h2>
  </div>
  <div class="grid-3">
    <div class="feature-card tilt-card"><div class="feature-icon">◆</div><h3>Multi-class classification</h3><p>A single model distinguishes between eczema, melanoma, acne and psoriasis.</p></div>
    <div class="feature-card tilt-card"><div class="feature-icon">⚡</div><h3>Real-time prediction</h3><p>Results return in under two seconds — the experience feels instant.</p></div>
    <div class="feature-card tilt-card"><div class="feature-icon">▦</div><h3>Personal dashboard</h3><p>Every scan, confidence score and trend is saved to your account.</p></div>
    <div class="feature-card tilt-card"><div class="feature-icon">🤖</div><h3>AI Bot assistant</h3><p>Ask questions about skin conditions or the project, anytime.</p></div>
    <div class="feature-card tilt-card"><div class="feature-icon">◈</div><h3>Privacy-focused</h3><p>Images are processed for the prediction only — never shared publicly.</p></div>
    <div class="feature-card tilt-card"><div class="feature-icon">▤</div><h3>Metric-backed evaluation</h3><p>Accuracy, precision, recall and F1-score validate real performance.</p></div>
  </div>
</section>

<section class="section container">
  <div class="section-head">
    <span class="section-tag">Under the hood</span>
    <h2 class="section-title">Technology stack</h2>
  </div>
  <div class="stack-strip">
    <span class="stack-pill">Python</span>
    <span class="stack-pill">TensorFlow</span>
    <span class="stack-pill">Keras</span>
    <span class="stack-pill">OpenCV</span>
    <span class="stack-pill">Flask + SQLite</span>
    <span class="stack-pill">HAM10000 dataset</span>
    <span class="stack-pill">Three.js</span>
  </div>
</section>

<footer class="container">
  <div class="footer-inner">
    <p>© 2026 DermaScan AI — Internship project, Bug Slayers team.</p>
    <div class="footer-links"><a href="about.html">About</a><a href="chatbot.html">AI Bot</a><a href="contact.html">Contact</a></div>
  </div>
</footer>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script src="js/site.js"></script>
<script>
const canvas = document.getElementById('hero3d');
if (canvas && window.THREE) {
  const renderer = new THREE.WebGLRenderer({ canvas, antialias: true, alpha: true });
  const scene = new THREE.Scene();
  const camera = new THREE.PerspectiveCamera(55, 1, 0.1, 100);
  camera.position.z = 5;

  const wireGeo = new THREE.IcosahedronGeometry(1.6, 1);
  const wireMat = new THREE.MeshBasicMaterial({ color: 0x4fe8c9, wireframe: true, transparent: true, opacity: 0.55 });
  const wireMesh = new THREE.Mesh(wireGeo, wireMat);
  scene.add(wireMesh);

  const particleGeo = new THREE.IcosahedronGeometry(1.9, 3);
  const particleMat = new THREE.PointsMaterial({ color: 0xffb970, size: 0.035 });
  const particles = new THREE.Points(particleGeo, particleMat);
  scene.add(particles);

  scene.add(new THREE.AmbientLight(0xffffff, 1));

  function resize() {
    const w = canvas.clientWidth, h = canvas.clientHeight;
    camera.aspect = w / h; camera.updateProjectionMatrix();
    renderer.setSize(w, h, false);
  }
  window.addEventListener('resize', resize);
  setTimeout(resize, 50);

  function animate() {
    requestAnimationFrame(animate);
    wireMesh.rotation.y += 0.004;
    wireMesh.rotation.x += 0.002;
    particles.rotation.y -= 0.0025;
    renderer.render(scene, camera);
  }
  animate();
}

document.querySelectorAll('.tilt-card').forEach(card => {
  card.addEventListener('mousemove', e => {
    const r = card.getBoundingClientRect();
    const x = (e.clientX - r.left) / r.width - 0.5;
    const y = (e.clientY - r.top) / r.height - 0.5;
    card.style.transform = `rotateY(${x * 14}deg) rotateX(${-y * 14}deg)`;
  });
  card.addEventListener('mouseleave', () => { card.style.transform = 'rotateY(0) rotateX(0)'; });
});
</script>
</body>
</html>


Overwriting dermascan/frontend/index.html


In [11]:
%%writefile dermascan/frontend/login.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Log in — DermaScan AI</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="css/style.css">
</head>
<body>

<div class="auth-shell">
  <div class="auth-visual">
    <a href="index.html" class="brand"><span class="brand-mark">DS</span> DermaScan AI</a>
    <div class="pitch">
      <h2>Every scan, tracked in one dashboard.</h2>
      <p>Log in to pick up where you left off — your scan history, condition breakdown, and confidence trends are all saved to your account.</p>
    </div>
    <div class="scan-frame" style="max-width:340px;">
      <div class="skin-swatch"></div>
      <div class="lesion"></div>
      <div class="scan-grid"></div>
      <div class="scan-line"></div>
      <div class="corner tl"></div><div class="corner tr"></div>
      <div class="corner bl"></div><div class="corner br"></div>
    </div>
  </div>

  <div class="auth-form-side">
    <div class="auth-card">
      <div class="kicker">Welcome back</div>
      <h1>Log in to your account</h1>
      <p class="switcher">New here? <a href="signup.html">Create an account</a></p>

      <div class="auth-error" id="authError"></div>

      <form id="loginForm">
        <div class="field">
          <label for="email">Email</label>
          <input type="email" id="email" required>
        </div>
        <div class="field">
          <label for="password">Password</label>
          <input type="password" id="password" required>
        </div>
        <button type="submit" class="btn btn-primary btn-block" id="loginBtn">Log in</button>
      </form>
    </div>
  </div>
</div>

<script src="js/auth.js"></script>
</body>
</html>


Overwriting dermascan/frontend/login.html


In [12]:
%%writefile dermascan/frontend/signup.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Sign up — DermaScan AI</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="css/style.css">
</head>
<body>

<div class="auth-shell">
  <div class="auth-visual">
    <a href="index.html" class="brand"><span class="brand-mark">DS</span> DermaScan AI</a>
    <div class="pitch">
      <h2>Start your first scan in under a minute.</h2>
      <p>Create a free account to get a personal dashboard that tracks every image you analyze and the condition breakdown over time.</p>
    </div>
    <div class="scan-frame" style="max-width:340px;">
      <div class="skin-swatch"></div>
      <div class="lesion"></div>
      <div class="scan-grid"></div>
      <div class="scan-line"></div>
      <div class="corner tl"></div><div class="corner tr"></div>
      <div class="corner bl"></div><div class="corner br"></div>
    </div>
  </div>

  <div class="auth-form-side">
    <div class="auth-card">
      <div class="kicker">Get started</div>
      <h1>Create your account</h1>
      <p class="switcher">Already have one? <a href="login.html">Log in</a></p>

      <div class="auth-error" id="authError"></div>

      <form id="signupForm">
        <div class="field">
          <label for="name">Full name</label>
          <input type="text" id="name" required>
        </div>
        <div class="field">
          <label for="email">Email</label>
          <input type="email" id="email" required>
        </div>
        <div class="field">
          <label for="password">Password</label>
          <input type="password" id="password" minlength="6" required>
        </div>
        <div class="field">
          <label for="confirmPassword">Confirm password</label>
          <input type="password" id="confirmPassword" minlength="6" required>
        </div>
        <button type="submit" class="btn btn-primary btn-block" id="signupBtn">Create account</button>
      </form>
    </div>
  </div>
</div>

<script src="js/auth.js"></script>
</body>
</html>


Overwriting dermascan/frontend/signup.html


In [13]:
%%writefile dermascan/frontend/dashboard.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Dashboard — DermaScan AI</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="css/style.css">
</head>
<body>

<div class="app-shell">
  <aside class="sidebar">
    <a href="index.html" class="brand"><span class="brand-mark">DS</span> DermaScan AI</a>
    <nav class="side-nav">
      <a href="dashboard.html" class="active"><span class="ic">▦</span> Overview</a>
      <a href="predict.html"><span class="ic">◆</span> New scan</a>
      <a href="about.html"><span class="ic">ⓘ</span> About the model</a>
      <a href="chatbot.html"><span class="ic">🤖</span> AI Bot</a>
      <a href="contact.html"><span class="ic">✉</span> Contact</a>
    </nav>
    <div class="sidebar-footer">
      <div class="sidebar-user">
        <div class="avatar" id="userAvatar">?</div>
        <div>
          <div class="name" id="userName">Loading…</div>
          <div class="email" id="userEmail"></div>
        </div>
      </div>
      <div class="logout-link" id="logoutBtn"><span class="ic">⏻</span> Log out</div>
    </div>
  </aside>

  <main class="app-main">
    <div class="app-topbar">
      <div>
        <h1 id="greeting">Welcome back</h1>
        <p>Here's a summary of your scan activity.</p>
      </div>
      <a href="predict.html" class="btn btn-primary">+ New scan</a>
    </div>

    <div class="stat-cards" id="statCards">
      <div class="stat-card"><div class="top"><span class="ic teal">◆</span></div><div class="value" id="statTotal">—</div><div class="label">Total scans</div></div>
      <div class="stat-card"><div class="top"><span class="ic amber">★</span></div><div class="value" id="statTop">—</div><div class="label">Most common result</div></div>
      <div class="stat-card"><div class="top"><span class="ic violet">%</span></div><div class="value" id="statAvgConf">—</div><div class="label">Average confidence</div></div>
      <div class="stat-card"><div class="top"><span class="ic danger">⚠</span></div><div class="value" id="statFlagged">—</div><div class="label">Melanoma flags</div></div>
    </div>

    <div class="dash-grid">
      <div class="panel">
        <h3>Condition breakdown</h3>
        <div id="donutContainer"></div>
      </div>
      <div class="panel">
        <h3>Recent scans</h3>
        <div class="scan-list" id="recentScans">
          <p class="empty-note">Loading…</p>
        </div>
      </div>
    </div>
  </main>
</div>

<script src="js/guard.js"></script>
<script src="js/dashboard.js"></script>
</body>
</html>


Overwriting dermascan/frontend/dashboard.html


In [14]:
%%writefile dermascan/frontend/predict.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>New scan — DermaScan AI</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="css/style.css">
</head>
<body>

<div class="app-shell">
  <aside class="sidebar">
    <a href="index.html" class="brand"><span class="brand-mark">DS</span> DermaScan AI</a>
    <nav class="side-nav">
      <a href="dashboard.html"><span class="ic">▦</span> Overview</a>
      <a href="predict.html" class="active"><span class="ic">◆</span> New scan</a>
      <a href="about.html"><span class="ic">ⓘ</span> About the model</a>
      <a href="chatbot.html"><span class="ic">🤖</span> AI Bot</a>
      <a href="contact.html"><span class="ic">✉</span> Contact</a>
    </nav>
    <div class="sidebar-footer">
      <div class="sidebar-user">
        <div class="avatar" id="userAvatar">?</div>
        <div>
          <div class="name" id="userName">Loading…</div>
          <div class="email" id="userEmail"></div>
        </div>
      </div>
      <div class="logout-link" id="logoutBtn"><span class="ic">⏻</span> Log out</div>
    </div>
  </aside>

  <main class="app-main">
    <div class="app-topbar">
      <div>
        <h1>New scan</h1>
        <p>Upload a clear, well-lit close-up of the affected skin area.</p>
      </div>
    </div>

    <div class="predict-layout">
      <div class="upload-card">
        <div class="dropzone" id="dropzone">
          <svg width="34" height="34" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.6"><path d="M12 16V4M12 4l-4 4M12 4l4 4"/><path d="M4 16v3a2 2 0 002 2h12a2 2 0 002-2v-3"/></svg>
          <p>Drag & drop an image, or click to browse</p>
          <p class="hint">JPG or PNG, up to 8MB</p>
          <input type="file" id="fileInput" accept="image/png, image/jpeg">
        </div>

        <div class="preview-wrap" id="previewWrap">
          <img id="previewImg" src="" alt="Uploaded skin image preview">
          <div class="scan-line"></div>
        </div>

        <div style="margin-top:22px; display:flex; gap:12px;">
          <button class="btn btn-primary" id="analyzeBtn" disabled>Analyze image</button>
          <button class="btn btn-ghost" id="resetBtn">Reset</button>
        </div>
      </div>

      <div class="result-card" id="resultCard">
        <div class="result-empty" id="resultEmpty">
          <svg width="40" height="40" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.3"><circle cx="12" cy="12" r="9"/><path d="M12 8v5M12 16h.01"/></svg>
          <p>No image analyzed yet.<br>Upload a photo and click "Analyze image" to see results here.</p>
        </div>
        <div id="resultContent" style="display:none;"></div>
      </div>
    </div>
  </main>
</div>

<script src="js/guard.js"></script>
<script src="js/predict.js"></script>
</body>
</html>


Overwriting dermascan/frontend/predict.html


In [15]:
%%writefile dermascan/frontend/about.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>About — DermaScan AI</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="css/style.css">
</head>
<body>

<nav class="navbar">
  <div class="nav-inner">
    <a href="index.html" class="brand"><span class="brand-mark">DS</span> DermaScan AI</a>
    <div class="nav-links">
      <a href="index.html">Home</a>
      <a href="about.html" class="active">About</a>
      <a href="chatbot.html">AI Bot</a>
      <a href="contact.html">Contact</a>
      <a href="login.html" class="nav-cta" id="authNavLink">Log in</a>
    </div>
  </div>
</nav>

<section class="section container">
  <div class="section-head">
    <span class="section-tag">The project</span>
    <h2 class="section-title">Why automated skin disease classification</h2>
    <p class="section-desc">Skin diseases affect millions globally, and early detection is central to effective treatment — but accurate diagnosis usually requires an expert dermatologist, which is slow and costly to access. DermaScan AI uses a convolutional neural network to give both clinicians and everyday users an instant first read from a dermoscopic image, saved to a personal dashboard for tracking over time.</p>
  </div>
</section>

<section class="section container">
  <div class="section-head">
    <span class="section-tag">Dataset</span>
    <h2 class="section-title">HAM10000</h2>
    <p class="section-desc">"Human Against Machine with 10000 training images" — a publicly available benchmark of labeled dermoscopic images, used here to train and validate the classifier.</p>
  </div>
  <div class="metric-grid">
    <div class="metric-card"><div class="num">10,015</div><div class="lbl">Labeled images</div></div>
    <div class="metric-card"><div class="num">4</div><div class="lbl">Target classes</div></div>
    <div class="metric-card"><div class="num">224²</div><div class="lbl">Input resolution</div></div>
    <div class="metric-card"><div class="num">80/20</div><div class="lbl">Train / validation split</div></div>
  </div>
</section>

<section class="section container">
  <div class="section-head">
    <span class="section-tag">Architecture</span>
    <h2 class="section-title">Model pipeline</h2>
    <p class="section-desc">Images are pre-processed with OpenCV, passed through a MobileNetV2 feature extractor (transfer learning), then a custom classification head trained for this task.</p>
  </div>
  <div class="arch-diagram">
    <div class="arch-block"><b>Input</b>224×224×3</div>
    <span class="arch-arrow">→</span>
    <div class="arch-block"><b>Preprocess</b>Resize / Normalize</div>
    <span class="arch-arrow">→</span>
    <div class="arch-block"><b>MobileNetV2</b>Frozen base</div>
    <span class="arch-arrow">→</span>
    <div class="arch-block"><b>GAP</b>Pooling</div>
    <span class="arch-arrow">→</span>
    <div class="arch-block"><b>Dense 128</b>ReLU + Dropout</div>
    <span class="arch-arrow">→</span>
    <div class="arch-block"><b>Softmax</b>4 classes</div>
  </div>
</section>

<section class="section container">
  <div class="section-head">
    <span class="section-tag">Outcomes</span>
    <h2 class="section-title">What this project delivers</h2>
  </div>
  <div class="timeline">
    <div class="timeline-item"><h4>Trained deep learning model</h4><p>A CNN capable of classifying skin diseases with measurable accuracy, precision, recall and F1-score.</p></div>
    <div class="timeline-item"><h4>Full-stack application</h4><p>Signup, login, an upload-and-diagnose flow, and a personal dashboard — not just a script.</p></div>
    <div class="timeline-item"><h4>Clinical relevance</h4><p>Demonstrates how AI can assist doctors with early-stage detection rather than replace diagnosis.</p></div>
    <div class="timeline-item"><h4>Research potential</h4><p>A foundation for further work in telemedicine deployment and academic research.</p></div>
  </div>
</section>

<section class="section container">
  <div class="section-head">
    <span class="section-tag">Team</span>
    <h2 class="section-title">Bug Slayers</h2>
  </div>
  <div class="team-grid">
    <div class="team-card tilt-card"><div class="team-avatar">M</div><h4>Monu</h4><span>AIDS · REC Chennai</span></div>
    <div class="team-card tilt-card"><div class="team-avatar">T2</div><h4>Teammate</h4><span>Bug Slayers</span></div>
    <div class="team-card tilt-card"><div class="team-avatar">T3</div><h4>Teammate</h4><span>Bug Slayers</span></div>
    <div class="team-card tilt-card"><div class="team-avatar">T4</div><h4>Teammate</h4><span>Bug Slayers</span></div>
  </div>
</section>

<footer class="container">
  <div class="footer-inner">
    <p>© 2026 DermaScan AI — Internship project, Bug Slayers team.</p>
    <div class="footer-links"><a href="index.html">Home</a><a href="contact.html">Contact</a></div>
  </div>
</footer>

<script src="js/site.js"></script>
<script>
document.querySelectorAll('.tilt-card').forEach(card => {
  card.addEventListener('mousemove', e => {
    const r = card.getBoundingClientRect();
    const x = (e.clientX - r.left) / r.width - 0.5;
    const y = (e.clientY - r.top) / r.height - 0.5;
    card.style.transform = `rotateY(${x * 12}deg) rotateX(${-y * 12}deg)`;
  });
  card.addEventListener('mouseleave', () => { card.style.transform = 'rotateY(0) rotateX(0)'; });
});
</script>
</body>
</html>


Overwriting dermascan/frontend/about.html


In [16]:
%%writefile dermascan/frontend/contact.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Contact — DermaScan AI</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="css/style.css">
</head>
<body>

<nav class="navbar">
  <div class="nav-inner">
    <a href="index.html" class="brand"><span class="brand-mark">DS</span> DermaScan AI</a>
    <div class="nav-links">
      <a href="index.html">Home</a>
      <a href="about.html">About</a>
      <a href="chatbot.html">AI Bot</a>
      <a href="contact.html" class="active">Contact</a>
      <a href="login.html" class="nav-cta" id="authNavLink">Log in</a>
    </div>
  </div>
</nav>

<section class="section container">
  <div class="section-head">
    <span class="section-tag">Get in touch</span>
    <h2 class="section-title">Questions about the project</h2>
    <p class="section-desc">Reach out about the model, the dataset, or collaborating on the next iteration.</p>
  </div>

  <div class="contact-layout">
    <div>
      <div class="info-list">
        <div class="info-item"><div class="ic">✉</div><div><h4>Email</h4><p>bugslayers.team@example.com</p></div></div>
        <div class="info-item"><div class="ic">🏛</div><div><h4>Institution</h4><p>Rajalakshmi Engineering College, Chennai</p></div></div>
        <div class="info-item"><div class="ic">◈</div><div><h4>Team</h4><p>Bug Slayers — AIDS Department</p></div></div>
      </div>
    </div>

    <div class="form-card">
      <form id="contactForm">
        <div class="field"><label for="name">Name</label><input type="text" id="name" required></div>
        <div class="field"><label for="email">Email</label><input type="email" id="email" required></div>
        <div class="field"><label for="message">Message</label><textarea id="message" rows="5" required></textarea></div>
        <button type="submit" class="btn btn-primary">Send message</button>
        <p class="form-status" id="formStatus">Message sent — we'll get back to you soon.</p>
      </form>
    </div>
  </div>
</section>

<footer class="container">
  <div class="footer-inner">
    <p>© 2026 DermaScan AI — Internship project, Bug Slayers team.</p>
    <div class="footer-links"><a href="index.html">Home</a><a href="about.html">About</a></div>
  </div>
</footer>

<script src="js/site.js"></script>
</body>
</html>


Overwriting dermascan/frontend/contact.html


In [17]:
%%writefile dermascan/frontend/chatbot.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>AI Bot — DermaScan AI</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="css/style.css">
</head>
<body>

<nav class="navbar">
  <div class="nav-inner">
    <a href="index.html" class="brand"><span class="brand-mark">DS</span> DermaScan AI</a>
    <div class="nav-links">
      <a href="index.html">Home</a>
      <a href="about.html">About</a>
      <a href="chatbot.html" class="active">AI Bot</a>
      <a href="contact.html">Contact</a>
      <a href="login.html" class="nav-cta" id="authNavLink">Log in</a>
    </div>
  </div>
</nav>

<section class="section container">
  <div class="section-head">
    <span class="section-tag">AI Bot</span>
    <h2 class="section-title">Ask about skin conditions or this project</h2>
    <p class="section-desc">A quick assistant for common questions — not a substitute for a dermatologist.</p>
  </div>

  <div class="chat-shell">
    <div class="chat-suggestions" id="suggestions">
      <div class="chat-chip" data-q="What is eczema?">What is eczema?</div>
      <div class="chat-chip" data-q="What are melanoma warning signs?">Melanoma warning signs?</div>
      <div class="chat-chip" data-q="How does the model work?">How does the model work?</div>
      <div class="chat-chip" data-q="How accurate is DermaScan?">How accurate is it?</div>
    </div>
    <div class="chat-window" id="chatWindow">
      <div class="chat-msg bot">Hi! I'm the DermaScan AI Bot. Ask me about eczema, melanoma, acne, psoriasis, or how this project works.</div>
    </div>
    <div class="chat-input-row">
      <input type="text" id="chatInput" placeholder="Type your question...">
      <button class="btn btn-primary" id="sendBtn">Send</button>
    </div>
  </div>
</section>

<footer class="container">
  <div class="footer-inner">
    <p>© 2026 DermaScan AI — Internship project, Bug Slayers team.</p>
    <div class="footer-links"><a href="index.html">Home</a><a href="about.html">About</a><a href="contact.html">Contact</a></div>
  </div>
</footer>

<script src="js/site.js"></script>
<script src="js/chatbot.js"></script>
</body>
</html>


Overwriting dermascan/frontend/chatbot.html


## Step 4 — Write the backend files (model, DB, Flask app with auth)

In [18]:
%%writefile dermascan/backend/model.py
"""
model.py
Defines the CNN architecture used for skin disease classification.

Approach: transfer learning on MobileNetV2 (pretrained on ImageNet).
Training a CNN completely from scratch needs a very large dataset to
generalize well; transfer learning gets solid accuracy on a dataset the
size of HAM10000 with far less training time - a good choice for a
college/internship project as well as a practical one.
"""

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
except ImportError:
    tf = None
    layers = None
    models = None

IMG_SIZE = 224
CLASS_NAMES = ["Eczema", "Melanoma", "Acne", "Psoriasis"]


def build_model(num_classes: int = len(CLASS_NAMES), fine_tune: bool = False):
    """
    Builds and returns the classification model.
    """
    if tf is None:
        raise RuntimeError("TensorFlow is required to build or train the model, but is not installed.")
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = fine_tune

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=fine_tune)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="dermascan_cnn")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


if __name__ == "__main__":
    if tf is not None:
        m = build_model()
        m.summary()
    else:
        print("[DermaScan] TensorFlow not installed. Running in mock/development mode.")


Overwriting dermascan/backend/model.py

In [19]:
%%writefile dermascan/backend/utils.py
"""
utils.py
Image pre-processing helpers shared by training and inference.
"""

import numpy as np
import cv2
from PIL import Image

IMG_SIZE = 224


def read_image_from_bytes(file_bytes: bytes) -> np.ndarray:
    """Decode uploaded file bytes into an RGB numpy array."""
    pil_img = Image.open(file_bytes).convert("RGB")
    return np.array(pil_img)


def preprocess_image(img_array: np.ndarray) -> np.ndarray:
    """
    Resize + light denoising with OpenCV, then shape into a model-ready
    batch of one image. Pixel scaling for MobileNetV2 happens inside the
    model itself (see model.py), so this only handles geometry/cleanup.
    """
    img = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img = cv2.bilateralFilter(img, d=5, sigmaColor=40, sigmaSpace=40)  # mild denoise, keeps edges
    img = img.astype("float32")
    return np.expand_dims(img, axis=0)  # shape: (1, 224, 224, 3)


Overwriting dermascan/backend/utils.py


In [20]:
%%writefile dermascan/backend/db.py
"""
db.py
Tiny SQLite layer for users + scan history. No ORM — plain sqlite3,
kept intentionally simple for a college project.
"""

import os
import sqlite3
import json
from datetime import datetime

DB_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "dermascan.db")


def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_conn()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL,
            password_hash TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS scans (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            prediction TEXT NOT NULL,
            confidence REAL NOT NULL,
            probabilities TEXT NOT NULL,
            created_at TEXT NOT NULL,
            FOREIGN KEY (user_id) REFERENCES users (id)
        )
    """)
    conn.commit()
    conn.close()
    print(f"[DermaScan] Database ready at {DB_PATH}")


def create_user(name: str, email: str, password_hash: str):
    conn = get_conn()
    cur = conn.execute(
        "INSERT INTO users (name, email, password_hash, created_at) VALUES (?, ?, ?, ?)",
        (name, email, password_hash, datetime.utcnow().isoformat()),
    )
    conn.commit()
    user_id = cur.lastrowid
    conn.close()
    return user_id


def get_user_by_email(email: str):
    conn = get_conn()
    row = conn.execute("SELECT * FROM users WHERE email = ?", (email,)).fetchone()
    conn.close()
    return dict(row) if row else None


def get_user_by_id(user_id: int):
    conn = get_conn()
    row = conn.execute("SELECT * FROM users WHERE id = ?", (user_id,)).fetchone()
    conn.close()
    return dict(row) if row else None


def create_scan(user_id: int, prediction: str, confidence: float, probabilities: dict):
    conn = get_conn()
    conn.execute(
        "INSERT INTO scans (user_id, prediction, confidence, probabilities, created_at) VALUES (?, ?, ?, ?, ?)",
        (user_id, prediction, confidence, json.dumps(probabilities), datetime.utcnow().isoformat()),
    )
    conn.commit()
    conn.close()


def get_dashboard_stats(user_id: int):
    conn = get_conn()
    rows = conn.execute(
        "SELECT prediction, confidence, created_at FROM scans WHERE user_id = ? ORDER BY created_at DESC",
        (user_id,),
    ).fetchall()
    conn.close()

    rows = [dict(r) for r in rows]
    total = len(rows)

    condition_counts = {"Eczema": 0, "Melanoma": 0, "Acne": 0, "Psoriasis": 0}
    for r in rows:
        if r["prediction"] in condition_counts:
            condition_counts[r["prediction"]] += 1

    most_common = max(condition_counts, key=condition_counts.get) if total else None
    avg_confidence = sum(r["confidence"] for r in rows) / total if total else 0
    melanoma_flags = condition_counts["Melanoma"]

    return {
        "total_scans": total,
        "condition_counts": condition_counts,
        "most_common": most_common,
        "avg_confidence": avg_confidence,
        "melanoma_flags": melanoma_flags,
        "recent": rows[:8],
    }


Overwriting dermascan/backend/db.py


In [21]:
%%writefile dermascan/backend/app.py
"""
app.py (v2)
Serves the DermaScan frontend and a small REST + session-auth API from
one Flask process — designed to run inside Colab behind an ngrok tunnel,
or locally.

Auth: simple session-cookie auth (Flask's built-in signed session,
werkzeug password hashing, SQLite for storage). This is fine for a
college project demo — it is NOT hardened for production use (no rate
limiting, no email verification, no HTTPS enforcement).

Endpoints:
  GET  /                      -> index.html
  POST /api/signup             {name, email, password}
  POST /api/login              {email, password}
  POST /api/logout
  GET  /api/me                 -> current session user, or 401
  POST /api/predict            multipart "image" (requires login) -> saves scan to history
  GET  /api/dashboard/stats    (requires login) -> totals, breakdown, recent scans
  POST /api/contact            {name, email, message}
  GET  /api/health
"""

import io
import os
import random

from flask import Flask, request, jsonify, session, send_from_directory
from flask_cors import CORS
from werkzeug.security import generate_password_hash, check_password_hash
import numpy as np

import db
from model import CLASS_NAMES, IMG_SIZE
from utils import read_image_from_bytes, preprocess_image

BACKEND_DIR = os.path.dirname(os.path.abspath(__file__))
FRONTEND_DIR = os.path.join(os.path.dirname(BACKEND_DIR), "frontend")

app = Flask(__name__, static_folder=FRONTEND_DIR, static_url_path="")
app.secret_key = os.environ.get("DERMASCAN_SECRET_KEY", "dev-only-change-me-for-real-use")
CORS(app, supports_credentials=True)

MODEL_PATH = os.path.join(BACKEND_DIR, "skin_model.h5")
_model = None
_using_mock = True


def load_model_if_available():
    global _model, _using_mock
    if os.path.exists(MODEL_PATH):
        import tensorflow as tf
        _model = tf.keras.models.load_model(MODEL_PATH)
        _using_mock = False
        print(f"[DermaScan] Loaded trained model from {MODEL_PATH}")
    else:
        _using_mock = True
        print("[DermaScan] No trained model found — serving MOCK predictions.")


def mock_predict(image_array: np.ndarray) -> np.ndarray:
    seed = int(np.sum(image_array)) % (2**32 - 1)
    rng = random.Random(seed)
    raw = [rng.uniform(0.05, 1.0) for _ in CLASS_NAMES]
    top_idx = raw.index(max(raw))
    raw[top_idx] *= 3.0
    total = sum(raw)
    return np.array([v / total for v in raw])


def login_required(fn):
    from functools import wraps
    @wraps(fn)
    def wrapper(*args, **kwargs):
        if "user_id" not in session:
            return jsonify({"error": "Not logged in"}), 401
        return fn(*args, **kwargs)
    return wrapper


# ---------- Frontend ----------
@app.route("/")
def home():
    return send_from_directory(FRONTEND_DIR, "index.html")


# ---------- Auth ----------
@app.route("/api/signup", methods=["POST"])
def signup():
    data = request.get_json(silent=True) or {}
    name = (data.get("name") or "").strip()
    email = (data.get("email") or "").strip().lower()
    password = data.get("password") or ""

    if not (name and email and password):
        return jsonify({"error": "Name, email and password are all required"}), 400
    if len(password) < 6:
        return jsonify({"error": "Password must be at least 6 characters"}), 400
    if db.get_user_by_email(email):
        return jsonify({"error": "An account with that email already exists"}), 409

    user_id = db.create_user(name, email, generate_password_hash(password))
    session["user_id"] = user_id
    return jsonify({"user": {"id": user_id, "name": name, "email": email}})


@app.route("/api/login", methods=["POST"])
def login():
    data = request.get_json(silent=True) or {}
    email = (data.get("email") or "").strip().lower()
    password = data.get("password") or ""

    user = db.get_user_by_email(email)
    if not user or not check_password_hash(user["password_hash"], password):
        return jsonify({"error": "Invalid email or password"}), 401

    session["user_id"] = user["id"]
    return jsonify({"user": {"id": user["id"], "name": user["name"], "email": user["email"]}})


@app.route("/api/logout", methods=["POST"])
def logout():
    session.clear()
    return jsonify({"status": "logged_out"})


@app.route("/api/me", methods=["GET"])
def me():
    if "user_id" not in session:
        return jsonify({"error": "Not logged in"}), 401
    user = db.get_user_by_id(session["user_id"])
    if not user:
        session.clear()
        return jsonify({"error": "Not logged in"}), 401
    return jsonify({"user": {"id": user["id"], "name": user["name"], "email": user["email"]}})


# ---------- Prediction (protected) ----------
@app.route("/api/predict", methods=["POST"])
@login_required
def predict():
    if "image" not in request.files:
        return jsonify({"error": "No image file provided under field 'image'"}), 400

    file = request.files["image"]
    try:
        img_array = read_image_from_bytes(io.BytesIO(file.read()))
    except Exception as e:
        return jsonify({"error": f"Could not read image: {e}"}), 400

    processed = preprocess_image(img_array)

    if _using_mock:
        probs = mock_predict(processed[0])
    else:
        preds = _model.predict(processed, verbose=0)
        probs = preds[0]

    top_idx = int(np.argmax(probs))
    prediction = CLASS_NAMES[top_idx]
    confidence = float(probs[top_idx])
    probabilities = {CLASS_NAMES[i]: float(probs[i]) for i in range(len(CLASS_NAMES))}

    db.create_scan(session["user_id"], prediction, confidence, probabilities)

    return jsonify({
        "prediction": prediction,
        "confidence": confidence,
        "probabilities": probabilities,
        "mode": "mock" if _using_mock else "trained_model",
    })


# ---------- Dashboard (protected) ----------
@app.route("/api/dashboard/stats", methods=["GET"])
@login_required
def dashboard_stats():
    stats = db.get_dashboard_stats(session["user_id"])
    return jsonify(stats)


# ---------- Contact (public) ----------
@app.route("/api/contact", methods=["POST"])
def contact():
    data = request.get_json(silent=True) or {}
    name = data.get("name", "").strip()
    email = data.get("email", "").strip()
    message = data.get("message", "").strip()
    if not (name and email and message):
        return jsonify({"error": "name, email and message are all required"}), 400
    print(f"[Contact] {name} <{email}>: {message}")
    return jsonify({"status": "received"})


@app.route("/api/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "mode": "mock" if _using_mock else "trained_model"})


if __name__ == "__main__":
    db.init_db()
    load_model_if_available()
    app.run(port=5000)


Overwriting dermascan/backend/app.py


## Step 5 — One-time ngrok signup (gives you a public link)
1. Sign up free at `https://dashboard.ngrok.com/signup`.
2. Copy your token from `https://dashboard.ngrok.com/get-started/your-authtoken` (or reuse one you already created).
3. Paste it below, replacing `PASTE_YOUR_TOKEN_HERE`, then run the cell.

In [22]:
try:
    from pyngrok import ngrok
    NGROK_AUTH_TOKEN = "3IuoLwTIaAom9mqhHw7KqZ33NMe_3xgdCAb7RnfyFZFW9k9Dq"
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("Ngrok auth token configured.")
except Exception as e:
    print(f"Ngrok note (running locally, tunnel not required): {e}")


Ngrok auth token configured.

## Step 6 — Launch the full website
Starts Flask, creates the SQLite database, and opens a public ngrok link. Click the link -- you land on the DermaScan homepage with the 3D hero animation. Click **Create free account**, sign up, and you're in your dashboard. Try **Diagnose** (works instantly in mock mode) and **AI Bot** (works instantly, no API key needed).

In [23]:
import sys, os, threading, time

# Add backend directory to sys.path (supports both local Windows and Google Colab)
backend_path = os.path.abspath('dermascan/backend' if os.path.exists('dermascan/backend') else '/content/dermascan/backend')
if backend_path not in sys.path:
    sys.path.append(backend_path)

import db
import app as backend_app

def _run():
    db.init_db()
    backend_app.load_model_if_available()
    backend_app.app.run(port=5000, use_reloader=False)

server_thread = threading.Thread(target=_run, daemon=True)
server_thread.start()
time.sleep(2)

print("\n==================================================")
print(" Your DermaScan AI website is live at:")
print(" Local URL: http://127.0.0.1:5000")
print("==================================================")

try:
    from pyngrok import ngrok
    public_url = ngrok.connect(5000)
    print(" Public ngrok URL:", public_url)
except Exception as e:
    print(f"(Running locally; ngrok tunnel skipped: {e})")


[DermaScan] Database ready at C:\Users\sivak\Desktop\Cybernaut Internship\DermaScan\dermascan\backend\dermascan.db
[DermaScan] No trained model found — serving MOCK predictions.
 * Serving Flask app 'app'


 * Debug mode: off



 * Running on http://127.0.0.1:5000


Press CTRL+C to quit


 Your DermaScan AI website is live at:
 Local URL: http://127.0.0.1:5000


 Public ngrok URL: NgrokTunnel: "https://backlog-fanciness-pretended.ngrok-free.dev" -> "http://localhost:5000"
